# Param M times

### Setup

In [ ]:
%%capture
%pip install numpy matplotlib statsmodels arch joblib

In [ ]:
import numpy as np, statsmodels.api as sm, warnings, matplotlib.pyplot as plt
from arch import arch_model
from joblib import Parallel, delayed
warnings.filterwarnings("ignore")

def winkler(y, m, v, a=0.05):
    d = 1.96 * np.sqrt(v); l, u = m-d, m+d
    return (u-l) + 40*max(l-y, 0) + 40*max(y-u, 0)

### Inputs

In [ ]:
np.random.seed(42)

N, M = 5000, 10
P, mu, phi, sig = [[0.98, 0.02], [0.05, 0.95]], [0.0, 0.0], [0.9, 0.2], [0.1, 1.5]

test_window = 50    # How many predictions to score
train_size = 400    # How many past points to train on per step

paths = []
for _ in range(M):
    s, y = [0], [0.]
    for t in range(1, N):
        s.append(np.random.choice([0, 1], p=P[s[-1]]))
        y.append(mu[s[-1]] + phi[s[-1]] * y[-1] + np.random.normal(0, sig[s[-1]]))
    paths.append(np.array(y))

### Paths

In [ ]:
plt.figure(figsize=(15, 2 * M))

for i, y in enumerate(paths):
    # Column 1: Price Level (Cumulative Sum)
    plt.subplot(M, 2, 2*i + 1)
    plt.plot(np.cumsum(y), lw=0.8, color='navy')
    plt.title(f"Path {i+1}: Price Level")
    if i < M - 1: plt.xticks([])
    
    # Column 2: Returns (The Volatility)
    plt.subplot(M, 2, 2*i + 2)
    plt.plot(y, lw=0.4, color='teal', alpha=0.7)
    plt.axhline(0, color='black', lw=0.5, ls='--')
    plt.title(f"Path {i+1}: Returns")
    if i < M - 1: plt.xticks([])

plt.tight_layout()
plt.show()

### Multi-path backtest

In [ ]:
from joblib import Parallel, delayed

sweeps = {
    "AR(1)": ("ar", 1), "AR(3)": ("ar", 3),
    "ARCH(1)": ("arch", 1), "ARCH(3)": ("arch", 3), 
    "GARCH(1,1)": ("garch", (1,1)), "GARCH(2,2)": ("garch", (2,2)),
    "MSAR-O1": ("ms", 1) 
}

all_results = {k: [] for k in sweeps}
recovered = []

def process_step(tr, act, sweeps, is_last_step):
    step_results = {}
    failed_models = []
    last_params = None
    for name, (m_type, val) in sweeps.items():
        try:
            if m_type == "ar":
                m = sm.tsa.AutoReg(tr, val).fit()
                step_results[name] = [m.predict(len(tr), len(tr))[0], m.sigma2]
            elif m_type in ["arch", "garch"]:
                p, q = (val, 0) if m_type == "arch" else val
                m = arch_model(tr, vol='Garch', p=p, q=q).fit(disp='off')
                f = m.forecast(horizon=1)
                step_results[name] = [f.mean.values[-1,0], f.variance.values[-1,0]]
            elif m_type == "ms":
                m = sm.tsa.MarkovAutoregression(tr, k_regimes=2, order=val, switching_variance=True).fit(disp=False)
                p_nxt = np.asarray(m.regime_transition).reshape(2,2) @ np.asarray(m.filtered_marginal_probabilities)[-1].flatten()
                lags = tr[-val:][::-1]
                mu_regimes = [m.params[2+r] + np.dot(m.params[4+r:4+2*val:2], lags) for r in range(2)]
                step_results[name] = [np.dot(p_nxt, mu_regimes), np.dot(p_nxt, m.params[-2:])]
                # ONLY save params for O1 so we compare apples to apples
                if is_last_step and name == "MSAR-O1": last_params = m.params
        except Exception as e:
            failed_models.append(name)
            continue
            
    return step_results, last_params, failed_models

print(f"🚀 Parallelizing sweep: {len(sweeps)} models across {M} paths...")
for p_idx, path in enumerate(paths):
    print(f"\n" + "="*50 + f"\n🔥 Processing Path {p_idx+1}/{M}...\n" + "="*50)
    
    results = Parallel(n_jobs=-1, verbose=10)(
        delayed(process_step)(path[-i-train_size:-i], path[-i], sweeps, i == 1) 
        for i in range(test_window, 0, -1)
    )
    
    path_failures = {k: 0 for k in sweeps}
    path_hist = {k: [] for k in sweeps} # Group by path!
    
    for step_res, params, fails in results:
        if params is not None: recovered.append(params)
        for f in fails: path_failures[f] += 1
        for name in sweeps:
            if name in step_res: path_hist[name].append(step_res[name])

    # Append the completed path to the master list
    for k in sweeps: all_results[k].append(path_hist[k])

    print(f"\n📊 Path {p_idx+1} Health Report (Out of {window} steps):")
    for mod, fails in path_failures.items():
        if fails == 0: print(f"  ✅ {mod}: 100% Success")
        else: print(f"  ❌ {mod}: Failed {fails} times")

### Analysis

In [ ]:
def analyze(y, p, v):
    y, p, v = np.array(y), np.array(p), np.array(v)
    s, rets = np.sqrt(v), np.sign(p) * y
    return [np.mean((y-p)**2), np.mean([winkler(yi, pi, vi) for yi, pi, vi in zip(y, p, v)]), 
            np.mean((y >= p-1.96*s) & (y <= p+1.96*s)), np.mean(np.sign(y) == np.sign(p)), 
            (np.mean(rets)/np.std(rets))*np.sqrt(252) if np.std(rets)>0 else 0, np.sum(rets)]

h = ["MSE", "Winkler", "Cov95", "HitRate", "Sharpe", "TotRet"]
print(f"{'MODEL':<12} | " + " | ".join(f"{col:<7}" for col in h))
print("-" * 75)

summary = {}
for mod in all_results:
    # Match the specific path's actuals to the specific path's predictions
    m_list = [analyze(paths[i][-window:], np.array(pv)[:,0], np.array(pv)[:,1]) 
              for i, pv in enumerate(all_results[mod]) if len(pv) == window]
    if not m_list: continue
    
    avg = np.mean(m_list, axis=0)
    summary[mod] = avg
    print(f"{mod:<12} | " + " | ".join(f"{v:<7.3f}" for v in avg))

print("\n" + "="*40 + "\nPARAM RECOVERY (True vs Est MSAR-O1)\n" + "="*40)
if recovered:
    rp = np.mean(recovered, axis=0)
    print(f"True Phi:  {phi} | Est: {rp[4:6].round(2)}")
    print(f"True Sig2: {[round(x**2, 3) for x in sig]} | Est: {rp[-2:].round(3)}")
else:
    print("MSAR-O1 failed to converge on the final step for all paths; no params recovered.")

### Viz

In [ ]:
plt.figure(figsize=(15, 3 * M))
for i, path in enumerate(paths):
    plt.subplot(M, 1, i + 1)
    # Plot the full return series
    plt.plot(path, color='teal', lw=0.4, alpha=0.7, label=f"Path {i+1}")
    # Highlight the test window (the last 50 points we backtested)
    plt.axvspan(len(path)-window, len(path), color='orange', alpha=0.2, label='Test Window')
    plt.title(f"Simulated Returns: Path {i+1}")
    plt.legend(loc='upper left')
    if i < M-1: plt.xticks([])

plt.tight_layout(); plt.show()